# Structured text values - Python

All 11 Python examples from [docs/text.md](https://platob.github.io/yggdryl/text/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and run on any Python 3 kernel with the package
installed:

```console
pip install yggdryl
```

In [ ]:
from yggdryl import json

quote = json.loads('{"symbol":"AAPL","price":12.5}')

assert quote["symbol"] == "AAPL"
assert list(quote) == ["symbol", "price"]
assert json.dumps(quote) == b'{"symbol":"AAPL","price":12.5}'

## What a value can be

In [ ]:
import math

from yggdryl import json

assert json.loads("null") is None

# Width is a wire detail; Python sees one int type.
assert json.loads(json.dumps(2**70)) == 2**70

# Floats keep their exact bits.
assert math.copysign(1.0, json.loads(json.dumps(-0.0))) == -1.0
assert math.isnan(json.loads(json.dumps(math.nan)))

# Bytes survive a text format.
assert json.loads(json.dumps(b"\x00\x01")) == b"\x00\x01"

## Reading a shape you do not control

In [ ]:
from yggdryl import json

order = json.loads('{"symbol":"AAPL","legs":[{"price":12},{"price":13}],"venue":null}')

assert order["legs"][1]["price"] == 13
assert list(order) == ["symbol", "legs", "venue"]
assert "venue" in order
assert len(order.items()) == 3

# A missing key takes the default; a present null is still None.
assert order.get("currency", "EUR") == "EUR"
assert order["venue"] is None

## Rebuilding a mapping

In [ ]:
from yggdryl import json

order = json.loads('{"symbol":"AAPL","venue":null}')

updated = {**order, "venue": "XPAR", "currency": "EUR"}
assert list(updated) == ["symbol", "venue", "currency"]
assert updated["venue"] == "XPAR"

trimmed = {key: value for key, value in updated.items() if key != "venue"}
assert list(trimmed) == ["symbol", "currency"]

# The rebuilt mapping still encodes.
assert json.dumps(trimmed) == b'{"symbol":"AAPL","currency":"EUR"}'

## A name is not a type

In [ ]:
from yggdryl import json, yaml

tagged = (
    '{"$yggdryl":{"version":1,"type":"tag","tag":"app:Trade",'
    '"value":{"symbol":"AAPL"}}}'
)
assert json.loads(tagged) == {
    "$yggdryl": {
        "version": 1,
        "type": "tag",
        "tag": "app:Trade",
        "value": {"symbol": "AAPL"},
    }
}

# A YAML application tag annotates a node; the node is what arrives.
assert yaml.loads("!app:Trade {symbol: AAPL}\n") == {"symbol": "AAPL"}

## Four formats, one surface

In [ ]:
from yggdryl import json, toml, yaml

quote = {"symbol": "AAPL"}

assert json.dumps(quote) == b'{"symbol":"AAPL"}'
assert toml.loads(toml.dumps(quote)) == quote
assert yaml.loads(yaml.dumps(quote)) == quote

# JSON and YAML hold many documents; TOML holds exactly one.
assert json.dumps_all([{"id": 1}, {"id": 2}]) == b'{"id":1}\n{"id":2}\n'
assert list(json.loads_all(b'{"id":1}\n{"id":2}\n')) == [{"id": 1}, {"id": 2}]
assert list(yaml.loads_all("a: 1\n---\na: 2\n")) == [{"a": 1}, {"a": 2}]
assert not hasattr(toml, "dumps_all")

## Laying out a dump

In [ ]:
from yggdryl import DataType, Field

field = Field("id", "int64", nullable=False)

# `indent` is the Python spelling, matching `json.dumps`.
assert "\n" not in field.to_json()
assert field.to_json(indent=2).startswith('{\n  "name": "id",')
assert field.to_yaml().startswith("name: id\n")

# Bytes change, meaning does not.
for indent in (None, 2, 4):
    assert Field.from_json(field.to_json(indent=indent)) == field
    assert Field.from_yaml(field.to_yaml(indent=indent)) == field
    assert Field.from_toml(field.to_toml(indent=indent)) == field

## Inferring the format

In [ ]:
from yggdryl import json, toml, yaml

# There is nothing to infer: the module you import is the format.
quote = {"symbol": "AAPL"}
assert json.loads('{"symbol":"AAPL"}') == quote
assert toml.loads('symbol = "AAPL"\n') == quote
assert yaml.loads("symbol: AAPL\n") == quote

## Bounds on untrusted input

In [ ]:
from yggdryl import json

# The default bound is enforced, not advisory.
assert json.loads("[" * 100 + "]" * 100) is not None

try:
    json.loads("[" * 200 + "]" * 200)
except ValueError as error:
    assert "nesting depth limit exceeded" in str(error)
else:
    raise AssertionError("the depth limit was not applied")

## Failures carry a byte position

In [ ]:
from yggdryl import json, toml

try:
    json.loads('{"symbol": ')
except ValueError as error:
    assert "invalid json data at byte 11" in str(error)

try:
    toml.loads("symbol = ")
except ValueError as error:
    assert "invalid toml data at byte 9" in str(error)

## Through a storage handle

In [ ]:
import pathlib
import tempfile

from yggdryl import json, toml

with tempfile.TemporaryDirectory() as directory:
    quote = pathlib.Path(directory) / "quote.json"
    json.dump({"symbol": "AAPL"}, quote)

    assert quote.read_bytes() == b'{"symbol":"AAPL"}'
    assert json.load(quote) == {"symbol": "AAPL"}

    # The suffix picks the reader; a str path works too.
    table = pathlib.Path(directory) / "quote.toml"
    toml.dump({"symbol": "AAPL"}, str(table))
    assert toml.load(table) == {"symbol": "AAPL"}